### Célula 1 — Calculando os KPIs para monitoramento

In [0]:
import pyspark.sql.functions as F
import builtins
from datetime import datetime

GOLD_PATH = "/Volumes/workspace/default/raw/gold/"

# Carregando a fato
f_tickets = spark.read.format("delta").load(f"{GOLD_PATH}f_customer_support_tickets/")

# Calculando KPIs
total        = f_tickets.count()
resolvidos   = f_tickets.filter(F.col("Is_Resolved") == 1).count()
backlog      = total - resolvidos
taxa         = builtins.round(resolvidos / total * 100, 1)

sat_media = builtins.round(
    f_tickets.filter(F.col("Customer_Satisfaction_Rating").isNotNull())
    .agg(F.avg("Customer_Satisfaction_Rating"))
    .collect()[0][0], 2
)

print("=" * 60)
print("MONITORAMENTO DE KPIs — CUSTOMER SUPPORT")
print(f"Executado em: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)
print(f"  Satisfação média:    {sat_media}")
print(f"  Taxa de resolução:   {taxa}%")
print(f"  Backlog aberto:      {backlog:,}")
print(f"  Total de tickets:    {total:,}")

MONITORAMENTO DE KPIs — CUSTOMER SUPPORT
Executado em: 2026-05-22 02:21:00
  Satisfação média:    2.99
  Taxa de resolução:   32.7%
  Backlog aberto:      5,700
  Total de tickets:    8,469


### Célula 2 — Definindo as regras de alerta

In [0]:
# Definindo metas e limites de alerta
METAS = {
    "satisfacao_media"  : {"meta": 3.0,  "operador": "<",  "descricao": "Satisfação média"},
    "taxa_resolucao_pct": {"meta": 70.0, "operador": "<",  "descricao": "Taxa de resolução"},
    "backlog_aberto"    : {"meta": 6000, "operador": ">",  "descricao": "Backlog aberto"},
}

# KPIs calculados
kpis = {
    "satisfacao_media"  : sat_media,
    "taxa_resolucao_pct": taxa,
    "backlog_aberto"    : backlog,
}

print("=" * 60)
print("VERIFICAÇÃO DE ALERTAS")
print("=" * 60)

alertas_disparados = []

for kpi, config in METAS.items():
    valor  = kpis[kpi]
    meta   = config["meta"]
    op     = config["operador"]
    desc   = config["descricao"]

    # Verifica condição
    dispara = (op == "<" and valor < meta) or (op == ">" and valor > meta)

    if dispara:
        msg = f"🔴 ALERTA — {desc}: {valor} (meta: {op} {meta})"
        alertas_disparados.append(msg)
        print(msg)
    else:
        print(f"✅ OK — {desc}: {valor} (meta: {op} {meta})")

print()
print("=" * 60)
if alertas_disparados:
    print(f"⚠️  {len(alertas_disparados)} ALERTA(S) DISPARADO(S)!")
else:
    print("✅ Todos os KPIs dentro das metas!")
print("=" * 60)

VERIFICAÇÃO DE ALERTAS
🔴 ALERTA — Satisfação média: 2.99 (meta: < 3.0)
🔴 ALERTA — Taxa de resolução: 32.7 (meta: < 70.0)
✅ OK — Backlog aberto: 5700 (meta: > 6000)

⚠️  2 ALERTA(S) DISPARADO(S)!


### Célula 3 — Enviando alerta por email

In [0]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from datetime import datetime

# ── Configurações — preencha com seus dados ──
EMAIL_REMETENTE = "seu_email@gmail.com"                              # seu email Gmail
EMAIL_SENHA     = "sua_senha_de_app"                                 # senha de app Gmail
EMAIL_DESTINO   = "seu_email@gmail.com"                              # destino do alerta

def enviar_alerta_email(alertas, kpis, metas):
    if not alertas:
        print("✅ Sem alertas — email não enviado.")
        return

    # Montando o corpo do email
    corpo_html = f"""
    <html><body>
    <h2 style="color:#c44e52">⚠️ Alertas — Customer Support BI</h2>
    <p>Executado em: <b>{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</b></p>
    <hr>
    <h3>KPIs com alerta:</h3>
    <ul>
    {''.join([f'<li style="color:#c44e52"><b>{a}</b></li>' for a in alertas])}
    </ul>
    <hr>
    <h3>Resumo completo:</h3>
    <table border="1" cellpadding="6" style="border-collapse:collapse">
      <tr style="background:#1E2130;color:white">
        <th>KPI</th><th>Valor Atual</th><th>Meta</th><th>Status</th>
      </tr>
      <tr>
        <td>Satisfação Média</td>
        <td>{kpis['satisfacao_media']}</td>
        <td>&gt;= 3.0</td>
        <td>{'🔴' if kpis['satisfacao_media'] < 3.0 else '✅'}</td>
      </tr>
      <tr>
        <td>Taxa de Resolução</td>
        <td>{kpis['taxa_resolucao_pct']}%</td>
        <td>&gt;= 70%</td>
        <td>{'🔴' if kpis['taxa_resolucao_pct'] < 70 else '✅'}</td>
      </tr>
      <tr>
        <td>Backlog Aberto</td>
        <td>{kpis['backlog_aberto']:,}</td>
        <td>&lt;= 6.000</td>
        <td>{'🔴' if kpis['backlog_aberto'] > 6000 else '✅'}</td>
      </tr>
    </table>
    <hr>
    <p style="color:#888">Customer Support BI — Databricks Pipeline</p>
    </body></html>
    """

    # Configurando o email
    msg = MIMEMultipart("alternative")
    msg["Subject"] = f"⚠️ [{len(alertas)} Alerta(s)] Customer Support BI — {datetime.now().strftime('%d/%m/%Y')}"
    msg["From"]    = EMAIL_REMETENTE
    msg["To"]      = EMAIL_DESTINO
    msg.attach(MIMEText(corpo_html, "html"))                         # formato HTML

    # Enviando
    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:     # Gmail SSL
            server.login(EMAIL_REMETENTE, EMAIL_SENHA)               # autenticação
            server.send_message(msg)                                 # envia
        print(f"✅ Email enviado para {EMAIL_DESTINO}!")
    except Exception as e:
        print(f"🔴 Erro ao enviar email: {e}")
        print("💡 Dica: use senha de app Gmail — não a senha normal")

# Chamando a função
enviar_alerta_email(alertas_disparados, kpis, METAS)

🔴 Erro ao enviar email: (535, b'5.7.8 Username and Password not accepted. For more information, go to\n5.7.8  https://support.google.com/mail/?p=BadCredentials af79cd13be357-914b5ff6479sm68062685a.23 - gsmtp')
💡 Dica: use senha de app Gmail — não a senha normal
